# Pipeline de avaliação — multi-modelo × multi-edital

Roda o agente em todas as combinações `(modelo, edital)` definidas abaixo, salva
um xlsx por combinação em `resultados/` e exibe um sumário consolidado.

**Idempotente:** se já existe um xlsx pra uma combinação, ela é pulada. Pra forçar
re-execução, apague o xlsx correspondente. Os timestamps no nome impedem que rodadas
antigas sejam sobrescritas — a checagem de existência usa glob por prefixo.

**Custo real vs estimado:**
- `custo_usd` (compatível com versão antiga do notebook) trata todo input como cache
  miss. É o teto.
- `custo_real_usd` separa cache hit de cache miss e aplica a tarifa correta. Esse é
  o número que vai bater com a fatura do provider, dentro de margem de erro.

Schema das colunas existentes (`id`, `categoria`, `pergunta`, `resposta`,
`input_tokens`, `output_tokens`, `total_tokens`, `custo_usd`, `latencia_s`, `erro`)
preservado pra não quebrar consumidores downstream tipo `eval_4omini/`.

In [1]:
import os, sys, time, glob
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_core.callbacks import BaseCallbackHandler

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## TokenTracker — captura tokens + cache

Lê três caminhos diferentes pra cobrir os providers:

1. `llm_output.token_usage` (formato OpenAI — também usado por DeepSeek via `ChatOpenAI` com `base_url` custom)
2. `usage_metadata` na message (formato nativo LangChain — usado por Anthropic e Google)

Em ambos os caminhos, tenta extrair quantos tokens de input bateram em cache:
- **DeepSeek:** `usage.prompt_cache_hit_tokens` (campo deles, no top-level do usage)
- **OpenAI:** `usage.prompt_tokens_details.cached_tokens`
- **Anthropic:** `usage_metadata.input_token_details.cache_read`

Pros providers que não populem nada disso, `cache_read_tokens` fica em zero — daí
`custo_real_usd` == `custo_usd` pra essas rodadas, o que é o comportamento correto.

In [2]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_tokens = 0    # input que bateu em cache (mais barato)
        self.cache_write_tokens = 0   # input gravado em cache (Anthropic; ignorado nos demais)
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        # --- Caminho 1: formato OpenAI (também DeepSeek via ChatOpenAI) ----------
        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens  += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0

            # DeepSeek: campo top-level
            ds_hit = usage.get("prompt_cache_hit_tokens") or 0
            # OpenAI: aninhado
            details = usage.get("prompt_tokens_details") or {}
            oai_hit = details.get("cached_tokens") or 0

            # Os dois nunca aparecem juntos na mesma resposta, então soma é seguro
            self.cache_read_tokens += (ds_hit + oai_hit)
            return

        # --- Caminho 2: usage_metadata nativo (Anthropic / Google) ---------------
        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens  += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0

                    details = um.get("input_token_details") or {}
                    self.cache_read_tokens  += details.get("cache_read", 0) or 0
                    self.cache_write_tokens += details.get("cache_creation", 0) or 0
                    break

## Configuração

**MODELOS:** comente os índices que não quer rodar agora. A pipeline pula sozinha
qualquer combinação que já tenha xlsx em `resultados/`.

**PRECOS:** três valores por modelo — `(input_miss, input_cached, output)` em USD/1M.
Pros modelos que não fazem cache no contexto desse agente, `input_cached` pode ser
igual a `input_miss` que não muda nada (já que `cache_read_tokens` vai ser 0).

In [3]:
MODELOS = [
    # (provider, modelo)
    # ("openai",    "gpt-4o-mini"),
    # ("openai",    "gpt-5.4-mini"),
    ("deepseek",  "deepseek-v4-flash"),
    # ("deepseek",  "deepseek-v4-pro"),
    # ("anthropic", "claude-haiku-4-5"),
    # --- a partir daqui, modelos caros — descomente quando for rodar ---
    # ("openai",    "gpt-5.4"),
    # ("anthropic", "claude-sonnet-4-6"),
    # ("anthropic", "claude-opus-4-7"),
    # ("openai",    "gpt-5.5"),
]

EDITAIS = [
    # (slug, edital_id no SQLite)
    ("bndes",     1),
    # ("cvm",       2),
    # ("petrobras", 3),
]

# USD por 1M tokens. Ordem: (input_miss, input_cached, output).
# Cache pricing:
#   - DeepSeek:  cache hit = 10% do input miss (90% off, automático)
#   - OpenAI:    cache hit = 50% do input miss (50% off, automático em prompts > ~1024 tokens)
#   - Anthropic: prompt caching exige cache_control nas mensagens; sem isso fica 0%.
#                Como o agent.py atual NÃO usa cache_control, cache_read será 0
#                pra todas as rodadas Anthropic. Os dois primeiros valores podem
#                ser iguais.
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.075,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  0.125,  2.00),
    "openai/gpt-5.4":                 (1.25,  0.625, 10.00),
    "openai/gpt-5.5":                 (5.00,  2.500, 30.00),
    "deepseek/deepseek-chat":         (0.28,  0.028,  0.42),
    "deepseek/deepseek-v4-flash":     (0.30,  0.030,  0.50),
    "deepseek/deepseek-v4-pro":       (0.435, 0.0435, 0.87),
    "anthropic/claude-haiku-4-5":     (1.00,  1.000,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00,  3.000, 15.00),
    "anthropic/claude-opus-4-7":      (5.00,  5.000, 25.00),
}

PERGUNTAS_DIR  = Path("perguntas")
RESULTADOS_DIR = Path("resultados")
RESULTADOS_DIR.mkdir(exist_ok=True)

print(f"Modelos ativos: {len(MODELOS)}")
print(f"Editais:        {len(EDITAIS)}")
print(f"Total de combos: {len(MODELOS) * len(EDITAIS)}")

Modelos ativos: 1
Editais:        1
Total de combos: 1


## Helpers

In [4]:
def slug(modelo: str) -> str:
    return modelo.replace("/", "_").replace(":", "_")


def ja_rodado(provider: str, modelo: str, edital_nome: str) -> str | None:
    """Retorna o caminho do xlsx mais recente pra essa combinação, se existir."""
    padrao = str(RESULTADOS_DIR / f"{edital_nome}__{provider}__{slug(modelo)}__*.xlsx")
    matches = sorted(glob.glob(padrao))
    return matches[-1] if matches else None


def custo_real(in_tok: int, out_tok: int, cache_read: int, p_in: float, p_in_cached: float, p_out: float) -> float:
    """Custo dividindo input em cache hit vs miss."""
    miss = max(in_tok - cache_read, 0)
    return (
        miss / 1_000_000 * p_in
        + cache_read / 1_000_000 * p_in_cached
        + out_tok / 1_000_000 * p_out
    )


def custo_estimado(in_tok: int, out_tok: int, p_in: float, p_out: float) -> float:
    """Teto: trata todo input como cache miss. Compatível com versão antiga."""
    return in_tok / 1_000_000 * p_in + out_tok / 1_000_000 * p_out


def rodar_combo(provider: str, modelo: str, edital_nome: str, edital_id: int) -> Path:
    """Roda o agente nas 50 perguntas de um edital, salva xlsx, retorna o caminho."""
    chave = f"{provider}/{modelo}"
    if chave not in PRECOS:
        raise ValueError(f"Preço de {chave} não definido em PRECOS")
    p_in, p_in_cached, p_out = PRECOS[chave]

    arq_perguntas = PERGUNTAS_DIR / f"{edital_nome}.xlsx"
    df_q = pd.read_excel(arq_perguntas)
    df_q = df_q.dropna(subset=["pergunta"])
    df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)

    agente = build_agent(provider=provider, model=modelo)
    tracker = TokenTracker()
    agente.llm       = agente.llm.with_config(callbacks=[tracker])
    agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

    resultados = []
    for i, row in df_q.iterrows():
        pergunta = row["pergunta"]
        t0 = time.time()
        erro, resposta = None, None

        tracker.reset()
        try:
            resposta = ask(
                agent=agente,
                question=pergunta,
                chat_history=[],
                edital_id_ativo=edital_id,
            )
        except Exception as e:
            erro = str(e)

        in_tok     = tracker.input_tokens
        out_tok    = tracker.output_tokens
        cache_read = tracker.cache_read_tokens
        n_calls    = tracker.n_calls
        latencia   = round(time.time() - t0, 2)

        resultados.append({
            "id":                row["id"],
            "categoria":         row.get("categoria", ""),
            "pergunta":          pergunta,
            "resposta":          resposta,
            "input_tokens":      in_tok,
            "output_tokens":     out_tok,
            "total_tokens":      in_tok + out_tok,
            "cache_read_tokens": cache_read,
            "cache_miss_tokens": max(in_tok - cache_read, 0),
            "n_invocacoes":      n_calls,
            "custo_usd":         round(custo_estimado(in_tok, out_tok, p_in, p_out), 6),
            "custo_real_usd":    round(custo_real(in_tok, out_tok, cache_read, p_in, p_in_cached, p_out), 6),
            "latencia_s":        latencia,
            "erro":              erro,
        })

        flag = "ERRO" if erro else "ok"
        cache_pct = (cache_read / in_tok * 100) if in_tok else 0
        print(f"  [{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
              f"calls={n_calls}  in={in_tok:>5} (cache {cache_pct:>4.0f}%)  out={out_tok:>4}")

    df_r = pd.DataFrame(resultados)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    arq = RESULTADOS_DIR / f"{edital_nome}__{provider}__{slug(modelo)}__{ts}.xlsx"
    df_r.to_excel(arq, index=False)
    return arq

## Pipeline

In [5]:
for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        # apaga xlsx antigos da mesma combinação (sobrescrita)
        for antigo in glob.glob(str(RESULTADOS_DIR / f"{edital_nome}__{provider}__{slug(modelo)}__*.xlsx")):
            Path(antigo).unlink()
            print(f"[del]  {Path(antigo).name}")

        print(f"[run]  {provider}/{modelo} × {edital_nome} (id={edital_id})")
        try:
            arq = rodar_combo(provider, modelo, edital_nome, edital_id)
            print(f"       salvo em {arq.name}")
        except Exception as e:
            print(f"       FALHOU: {e}")

print("\nPipeline concluída.")

[run]  deepseek/deepseek-v4-flash × bndes (id=1)
Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48291.31it/s]


  [ 1/50] ERRO    8.2s  calls=1  in= 2199 (cache  198%)  out= 150
  [ 2/50] ERRO   3.98s  calls=1  in= 2200 (cache  198%)  out= 137
  [ 3/50] ERRO   3.59s  calls=1  in= 2203 (cache  198%)  out= 124
  [ 4/50] ERRO   3.65s  calls=1  in= 2205 (cache  197%)  out= 128
  [ 5/50] ERRO   3.68s  calls=1  in= 2205 (cache  197%)  out= 150
  [ 6/50] ERRO   3.51s  calls=1  in= 2205 (cache  197%)  out= 149
  [ 7/50] ERRO    3.5s  calls=1  in= 2207 (cache  197%)  out= 156
  [ 8/50] ERRO   3.24s  calls=1  in= 2201 (cache  198%)  out= 129
  [ 9/50] ERRO    3.4s  calls=1  in= 2202 (cache  198%)  out= 148
  [10/50] ERRO   2.97s  calls=1  in= 2208 (cache  197%)  out= 105
  [11/50] ERRO   3.59s  calls=1  in= 2203 (cache  198%)  out= 142
  [12/50] ERRO   3.46s  calls=1  in= 2200 (cache  198%)  out= 140
  [13/50] ERRO    3.1s  calls=1  in= 2200 (cache  198%)  out= 141
  [14/50] ERRO   3.47s  calls=1  in= 2205 (cache  197%)  out= 150
  [15/50] ERRO   3.36s  calls=1  in= 2204 (cache  197%)  out= 137
  [16/50] 

## Sumário consolidado

Carrega todos os xlsx em `resultados/`, faz agregado por (modelo × edital). Inclui
tudo que está na pasta — não só o que rodou agora — então serve pra acompanhar o
estado geral conforme você for completando combinações.

In [6]:
import re

padrao_nome = re.compile(r"^(?P<edital>[^_]+)__(?P<provider>[^_]+)__(?P<modelo>.+)__\d{8}_\d{6}\.xlsx$")

linhas = []
for arq in sorted(RESULTADOS_DIR.glob("*.xlsx")):
    m = padrao_nome.match(arq.name)
    if not m:
        continue
    df = pd.read_excel(arq)

    # Compatibilidade com xlsx antigos (sem cache_read_tokens nem custo_real_usd)
    if "cache_read_tokens" not in df.columns:
        df["cache_read_tokens"] = 0
    if "custo_real_usd" not in df.columns:
        df["custo_real_usd"] = df["custo_usd"]

    linhas.append({
        "edital":           m["edital"],
        "provider":         m["provider"],
        "modelo":           m["modelo"],
        "n_perguntas":      len(df),
        "n_erros":          int(df["erro"].notna().sum()),
        "input_tokens":     int(df["input_tokens"].sum()),
        "output_tokens":    int(df["output_tokens"].sum()),
        "cache_read":       int(df["cache_read_tokens"].sum()),
        "cache_pct":        float(df["cache_read_tokens"].sum() / max(df["input_tokens"].sum(), 1)),
        "custo_estim_usd":  round(float(df["custo_usd"].sum()), 4),
        "custo_real_usd":   round(float(df["custo_real_usd"].sum()), 4),
        "latencia_med_s":   round(float(df["latencia_s"].mean()), 2),
        "latencia_p95_s":   round(float(df["latencia_s"].quantile(0.95)), 2),
        "arquivo":          arq.name,
    })

df_sumario = pd.DataFrame(linhas).sort_values(["edital", "custo_real_usd"]).reset_index(drop=True)
df_sumario["cache_pct"] = df_sumario["cache_pct"].map("{:.1%}".format)
df_sumario

,edital,provider,modelo,n_perguntas,n_erros,input_tokens,output_tokens,cache_read,cache_pct,custo_estim_usd,custo_real_usd,latencia_med_s,latencia_p95_s,arquivo
0,bndes,deepseek,deepseek-v4-flash,50,50,110224,7090,217600,197.4%,0.0366,0.0101,3.6,3.99,bndes__deepseek__deepseek-v4-flash__20260429_2...


## Tabela cruzada — custo real por modelo × edital

Pivot pra leitura rápida. Linha = modelo, coluna = edital.

In [7]:
df_pivot = (
    df_sumario.assign(modelo_full=lambda d: d["provider"] + "/" + d["modelo"])
              .pivot_table(
                  index="modelo_full",
                  columns="edital",
                  values="custo_real_usd",
                  aggfunc="last",   # se tiver duplicatas (mais de uma rodada), pega a mais recente
              )
)
df_pivot["total"] = df_pivot.sum(axis=1)
df_pivot.sort_values("total")

edital,bndes,total
modelo_full,,
deepseek/deepseek-v4-flash,0.0101,0.0101
